# Section e — Plant Profitability Model and Stranding Mechanism
## Notebook for profitability, negative margins, and stranding flags

This notebook is designed to support **Section e** of the report.

### Main goals
1. Load the processed plant-level dataset and the NGFS scenario dataset  
2. Build a plant × year × scenario panel  
3. Compute the operating margin of each plant  
4. Construct a negative-margin indicator  
5. Compute a stranding flag based on persistent losses  
6. Produce a few clean outputs for the report

### Main inputs
According to the database overview, this notebook uses:

- `data/processed/gppd_eu_metrics.csv`
- `data/processed/ngfs_scenarios_elec.csv`

### Main outputs
This notebook will produce:
- a plant-level profitability panel,
- a negative-margin indicator,
- a stranding flag,
- summary tables by fuel, scenario and year,
- a small set of figures that can feed Section e or the appendix.


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Robust project root detection
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "figures" / "section_e"
RES_DIR = PROJECT_ROOT / "results" / "section_e"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

PLANTS_PATH = DATA_DIR / "gppd_eu_metrics.csv"
SCEN_PATH = DATA_DIR / "ngfs_scenarios_elec.csv"

print("Project root:", PROJECT_ROOT)
print("Plants path exists:", PLANTS_PATH.exists(), PLANTS_PATH)
print("Scenarios path exists:", SCEN_PATH.exists(), SCEN_PATH)


## 2. Load data

In [ ]:
plants = pd.read_csv(PLANTS_PATH)
scen = pd.read_csv(SCEN_PATH)

print("plants shape:", plants.shape)
print("scenarios shape:", scen.shape)

plants.head()


In [ ]:
scen.head()


## 3. Quick checks

The profitability model requires:
- plant-level fuel category,
- plant-level carbon intensity,
- plant-level variable O&M cost,
- scenario-year electricity price,
- scenario-year carbon price,
- scenario-year fuel prices.


In [ ]:
required_plant_cols = [
    "gppd_idnr",
    "name",
    "country",
    "fuel",
    "capacity_mw",
    "commissioning_year",
    "lifetime",
    "carbon_intensity_tCO2_per_MWh_elec",
    "om_variable_eur2024_per_MWh",
]

required_scen_cols = ["scenario", "variable", "year", "value"]

missing_plant = [c for c in required_plant_cols if c not in plants.columns]
missing_scen = [c for c in required_scen_cols if c not in scen.columns]

print("Missing plant columns:", missing_plant if missing_plant else "None")
print("Missing scenario columns:", missing_scen if missing_scen else "None")
print("\nScenarios available:", sorted(scen["scenario"].unique()))
print("\nYears available:", scen["year"].min(), "to", scen["year"].max())
print("\nVariables available:")
print(sorted(scen["variable"].unique()))


## 4. Build a clean scenario price table

According to the data documentation, the following variables are directly relevant for the operating margin:
- `Price|Carbon`
- `Price|Secondary Energy|Electricity`
- `Price|Primary Energy|Coal`
- `Price|Primary Energy|Gas`
- `Price|Primary Energy|Oil`

Fuel prices in this file are already converted to **EUR2024/MWh of electricity**, so they are ready to be used in the profitability equation.


In [ ]:
price_vars = [
    "Price|Carbon",
    "Price|Secondary Energy|Electricity",
    "Price|Primary Energy|Coal",
    "Price|Primary Energy|Gas",
    "Price|Primary Energy|Oil",
]

prices = scen.loc[scen["variable"].isin(price_vars), ["scenario", "year", "variable", "value"]].copy()

price_wide = prices.pivot_table(
    index=["scenario", "year"],
    columns="variable",
    values="value",
    aggfunc="first"
).reset_index()

price_wide = price_wide.rename(columns={
    "Price|Carbon": "carbon_price_eur_per_tCO2",
    "Price|Secondary Energy|Electricity": "electricity_price_eur_per_MWh",
    "Price|Primary Energy|Coal": "fuel_price_coal_eur_per_MWh",
    "Price|Primary Energy|Gas": "fuel_price_gas_eur_per_MWh",
    "Price|Primary Energy|Oil": "fuel_price_oil_eur_per_MWh",
})

price_wide.head()


## 5. Expand to a plant × scenario × year panel

We now create the main analytical panel. Each row corresponds to:
- one plant,
- one scenario,
- one year.


In [ ]:
scenario_years = price_wide[["scenario", "year"]].drop_duplicates().copy()
scenario_years["__key"] = 1
plants_panel = plants.copy()
plants_panel["__key"] = 1

panel = plants_panel.merge(scenario_years, on="__key", how="inner").drop(columns="__key")
panel = panel.merge(price_wide, on=["scenario", "year"], how="left")

print(panel.shape)
panel.head()


## 6. Compute the components of the operating margin

We implement the following logic:

- electricity revenue = scenario electricity price,
- fuel cost = scenario fuel price corresponding to the plant fuel,
- O&M cost = plant-level variable O&M,
- carbon cost = carbon price × plant carbon intensity.


In [ ]:
panel["fuel_cost_eur_per_MWh"] = np.select(
    [
        panel["fuel"].eq("coal"),
        panel["fuel"].eq("gas"),
        panel["fuel"].eq("oil"),
    ],
    [
        panel["fuel_price_coal_eur_per_MWh"],
        panel["fuel_price_gas_eur_per_MWh"],
        panel["fuel_price_oil_eur_per_MWh"],
    ],
    default=np.nan
)

panel["om_cost_eur_per_MWh"] = panel["om_variable_eur2024_per_MWh"]

panel["carbon_cost_eur_per_MWh"] = (
    panel["carbon_price_eur_per_tCO2"] *
    panel["carbon_intensity_tCO2_per_MWh_elec"]
)

panel["operating_margin_eur_per_MWh"] = (
    panel["electricity_price_eur_per_MWh"]
    - panel["fuel_cost_eur_per_MWh"]
    - panel["om_cost_eur_per_MWh"]
    - panel["carbon_cost_eur_per_MWh"]
)

panel[[
    "gppd_idnr", "name", "fuel", "scenario", "year",
    "electricity_price_eur_per_MWh",
    "fuel_cost_eur_per_MWh",
    "om_cost_eur_per_MWh",
    "carbon_cost_eur_per_MWh",
    "operating_margin_eur_per_MWh"
]].head()


## 7. Negative-margin indicator

A plant-year-scenario observation is considered distressed when the operating margin is negative.


In [ ]:
panel["negative_margin_flag"] = (panel["operating_margin_eur_per_MWh"] < 0).astype(int)

panel[["scenario", "year", "fuel", "operating_margin_eur_per_MWh", "negative_margin_flag"]].head()


## 8. Stranding flag based on persistent losses

Baseline rule:
- a plant is flagged as stranded if it records negative operating margins for **K consecutive years**.

We use:
- `K = 3` as baseline.


In [ ]:
K = 3

panel = panel.sort_values(["gppd_idnr", "scenario", "year"]).reset_index(drop=True)

panel["rolling_negative_sum"] = (
    panel.groupby(["gppd_idnr", "scenario"])["negative_margin_flag"]
         .transform(lambda s: s.rolling(K, min_periods=K).sum())
)

panel["stranded_flag"] = (panel["rolling_negative_sum"] == K).astype(int)

panel[[
    "gppd_idnr", "name", "fuel", "scenario", "year",
    "operating_margin_eur_per_MWh", "negative_margin_flag",
    "rolling_negative_sum", "stranded_flag"
]].head(15)


## 9. Quick diagnostic tables

These tables are useful for Section e.


In [ ]:
summary_margin = (
    panel.groupby(["scenario", "year", "fuel"], as_index=False)
         .agg(
             avg_margin_eur_per_MWh=("operating_margin_eur_per_MWh", "mean"),
             median_margin_eur_per_MWh=("operating_margin_eur_per_MWh", "median"),
             share_negative_margin_pct=("negative_margin_flag", lambda x: 100 * x.mean()),
             share_stranded_pct=("stranded_flag", lambda x: 100 * x.mean()),
             plants=("gppd_idnr", "count"),
         )
)

summary_margin.head(12)


In [ ]:
summary_margin.to_csv(RES_DIR / "summary_margin_by_scenario_year_fuel.csv", index=False)
print("Saved:", RES_DIR / "summary_margin_by_scenario_year_fuel.csv")


## 10. Horizon tables for the report

We extract a compact table for a few key horizons.


In [ ]:
horizons = [2030, 2040, 2050]

horizon_table = (
    summary_margin.loc[summary_margin["year"].isin(horizons)]
    .sort_values(["scenario", "year", "fuel"])
    .reset_index(drop=True)
)

horizon_table


In [ ]:
horizon_table.to_csv(RES_DIR / "horizon_table_2030_2040_2050.csv", index=False)
print("Saved:", RES_DIR / "horizon_table_2030_2040_2050.csv")


## 11. Figure 1 — Margin trajectories by fuel and scenario

This figure is useful to show how profitability evolves across fuels under different transition paths.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for (scenario, fuel), grp in summary_margin.groupby(["scenario", "fuel"]):
    ax.plot(
        grp["year"],
        grp["avg_margin_eur_per_MWh"],
        label=f"{scenario} - {fuel}"
    )

ax.axhline(0, linewidth=1, linestyle="--")
ax.set_title("Average operating margin by fuel and scenario")
ax.set_xlabel("Year")
ax.set_ylabel("EUR/MWh")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

fig.savefig(FIG_DIR / "avg_margin_by_fuel_scenario.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Figure 2 — Share of negative-margin plants

This figure is useful to document when loss-making plants become widespread.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for (scenario, fuel), grp in summary_margin.groupby(["scenario", "fuel"]):
    ax.plot(
        grp["year"],
        grp["share_negative_margin_pct"],
        label=f"{scenario} - {fuel}"
    )

ax.set_title("Share of plants with negative operating margin")
ax.set_xlabel("Year")
ax.set_ylabel("Percent of plants")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

fig.savefig(FIG_DIR / "share_negative_margin_by_fuel_scenario.png", dpi=300, bbox_inches="tight")
plt.show()


## 13. Figure 3 — Share of stranded plants

This figure shows the dynamic effect of persistent losses.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for (scenario, fuel), grp in summary_margin.groupby(["scenario", "fuel"]):
    ax.plot(
        grp["year"],
        grp["share_stranded_pct"],
        label=f"{scenario} - {fuel}"
    )

ax.set_title(f"Share of stranded plants by fuel and scenario (K = {K})")
ax.set_xlabel("Year")
ax.set_ylabel("Percent of plants")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

fig.savefig(FIG_DIR / "share_stranded_by_fuel_scenario.png", dpi=300, bbox_inches="tight")
plt.show()


## 14. Optional distribution plot for a selected horizon

This can feed the empirical discussion of Section e.4.


In [ ]:
selected_year = 2040

plot_df = panel.loc[panel["year"].eq(selected_year)].copy()

fuel_order = ["coal", "gas", "oil"]
scenario_order = sorted(plot_df["scenario"].unique())

fig, axes = plt.subplots(len(scenario_order), 1, figsize=(8, 10), sharex=True)

if len(scenario_order) == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenario_order):
    temp = plot_df.loc[plot_df["scenario"].eq(scenario)]
    data = [temp.loc[temp["fuel"].eq(f), "operating_margin_eur_per_MWh"].dropna() for f in fuel_order]
    ax.boxplot(data, tick_labels=fuel_order)
    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_title(f"Operating margin distribution in {selected_year} — {scenario}")
    ax.set_ylabel("EUR/MWh")
    ax.grid(axis="y", alpha=0.3)

axes[-1].set_xlabel("Fuel")
plt.tight_layout()

fig.savefig(FIG_DIR / f"margin_distribution_{selected_year}.png", dpi=300, bbox_inches="tight")
plt.show()


## 15. Export the main analytical panel

This is the key output of the notebook. It can later be used for:
- country aggregation,
- portfolio mapping,
- sensitivity analysis.


In [ ]:
export_cols = [
    "gppd_idnr", "name", "country", "country_long", "fuel", "capacity_mw",
    "commissioning_year", "lifetime",
    "carbon_intensity_tCO2_per_MWh_elec",
    "om_variable_eur2024_per_MWh",
    "scenario", "year",
    "electricity_price_eur_per_MWh",
    "fuel_cost_eur_per_MWh",
    "om_cost_eur_per_MWh",
    "carbon_cost_eur_per_MWh",
    "operating_margin_eur_per_MWh",
    "negative_margin_flag",
    "rolling_negative_sum",
    "stranded_flag",
]

profitability_panel = panel[export_cols].copy()
profitability_panel.to_csv(RES_DIR / "plant_profitability_panel.csv", index=False)

print("Saved:", RES_DIR / "plant_profitability_panel.csv")
print("Final panel shape:", profitability_panel.shape)


## 16. Suggested interpretation blocks for Section e.4

Once the code has run, this notebook should allow you to answer questions such as:
- Which fuel becomes loss-making first?
- Under which scenario do margins deteriorate fastest?
- Is coal structurally more exposed than gas?
- How quickly do persistent losses translate into stranding flags?
- Are there major differences between 2030, 2040 and 2050?

These are the key empirical questions for the final text of Section e.4.
